# ACP Lecture Workbook — *Feature Selection & Parametrics*
### …ending at the GPT you trained last week

**The walkthrough notebook for the talk.** Six acts, each led by figures
generated from `../data/cpsc_merged.csv` or first-principles simulation —
nothing is clip-art, nothing asserted that isn't computed.

| Act | Question it answers |
| --- | --- |
| 1 | How does text become geometry — machine-readable? |
| 2 | Why do we regularize our features, and how does it work? |
| 3 | Which features matter? |
| 4 | How do we tune a model — honestly? |
| 5 | What is a neural network, really? |
| 6 | **A CPSC GPT** — the class's model, retrained on our own incident narratives |

> [!warning] Before the talk
> **Run All locally.** Act bodies lead with ⚙ scikit-learn cells (current
> idioms — what you'd write at work); every `wb_*` figure is embedded as an
> image and regenerated by the **appendix** (numpy first-principles receipts).
> The one offline GPT job is **done (2026-07-13)**:
> `gpt_train_instrumented.py --from-cpsc` → 29,077,970 chars, vocab 109 — it
> lights up Act 6's real loss curve, embedding table, and generated samples.
> Confirm the exact parameter count and AdamW decay from `cpsc_meta.txt`.
> Quoted CV numbers verified 2026-07-01/02 (sklearn 1.9) and 2026-07-10 (numpy
> first-principles; see `briefing-computations.ipynb`). Figures land in
> `../figures/wb_*.png` for the deck.

In [ ]:
# setup: style + save helper (colorblind-safe Okabe-Ito, back-row fonts)
import re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle
from collections import Counter
from pathlib import Path

OK = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]
plt.rcParams.update({"font.size": 14, "axes.titlesize": 17, "axes.labelsize": 15,
                     "figure.facecolor": "white"})
FIGDIR = Path("../figures"); FIGDIR.mkdir(exist_ok=True)
def SAVE(fig, name):
    fig.savefig(FIGDIR / f"wb_{name}.png", dpi=150, bbox_inches="tight")
rng = np.random.default_rng(0)
def sigmoid(z): return 1 / (1 + np.exp(-z))

In [ ]:
# shared data layer (numpy only): presence matrix on top-1000-by-df vocab,
# idf weights, top-12 hazard target, hand-built chi2 + macro-F1
df = pd.read_csv("../data/cpsc_merged.csv",
                 usecols=["incident_description", "product_1_hazard"]).dropna()
tokre = re.compile(r"\b[a-z]{2,}\b")
docs = [set(tokre.findall(t.lower())) for t in df["incident_description"]]
dfreq = Counter()
for s in docs: dfreq.update(s)
vocab = [w for w, _ in dfreq.most_common(1000)]
w2i = {w: i for i, w in enumerate(vocab)}
X = np.zeros((len(docs), 1000), dtype=np.float32)
for i, s in enumerate(docs):
    for w in s:
        j = w2i.get(w)
        if j is not None: X[i, j] = 1.0
N = len(docs)
idf = np.log(N / X.sum(0))
Xw = X * idf                                    # idf-weighted presence
yfire = df["product_1_hazard"].str.contains("Fire", case=False).values
top12 = df["product_1_hazard"].value_counts().head(12)
m12 = df["product_1_hazard"].isin(top12.index).values
classes12 = np.unique(df.loc[m12, "product_1_hazard"].values)
y12 = np.searchsorted(classes12, df.loc[m12, "product_1_hazard"].values)

def chi2_scores(Xb, yb):
    """chi2 of binary feature vs binary target (presence cells), hand-built —
    matches sklearn.feature_selection.chi2 on 0/1 features (verified Briefing 1)."""
    n1 = Xb[yb].sum(0); n0 = Xb[~yb].sum(0)
    N1, N0 = yb.sum(), (~yb).sum()
    tot = n1 + n0
    e1 = tot * N1 / len(yb); e0 = tot * N0 / len(yb)
    return (n1 - e1)**2 / e1 + (n0 - e0)**2 / e0

def macro_f1(yt, yp, K=12):
    f = []
    for k in range(K):
        tp = ((yp == k) & (yt == k)).sum(); fp = ((yp == k) & (yt != k)).sum()
        fn = ((yp != k) & (yt == k)).sum()
        P = tp/(tp+fp) if tp+fp else 0.0; R = tp/(tp+fn) if tp+fn else 0.0
        f.append(2*P*R/(P+R) if P+R else 0.0)
    return np.mean(f)

print(f"{N:,} incident reports | vocab kept: 1,000 | "
      f"top-12 hazards cover {top12.sum()/len(df):.1%}")

---
# Act 1 · How text becomes geometry — machine-readable
A computer cannot read. Before *any* model — logistic regression or GPT — text
must become **numbers**. The classical way: one column per word (TF-IDF). The
frontier way: each word/character becomes a **learned vector** (an embedding).
This act shows both are *coordinates in a space* — and that **distance in that
space = similarity in meaning**. That space is called **latent space**.

*Say on stage:* "every claim in the next three figures is computed from our own
CPSC data in the cell above it — nothing is stock art." 

In [ ]:
# ⚙ Act 1 lead — text -> numbers, the production way (also defines the talk's shared objects)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import StratifiedKFold, cross_val_score

d12 = df[m12]                                     # the 12-hazard subset
X_text, y_lab = d12["incident_description"], d12["product_1_hazard"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

tfidf = TfidfVectorizer(min_df=2)
Xt = tfidf.fit_transform(X_text)                  # text is now numbers — one line
print(f"TF-IDF: {Xt.shape[0]:,} reports x {Xt.shape[1]:,} word-features")

lsa = TruncatedSVD(n_components=2, random_state=0).fit(Xt)   # Act 1's map — one line
print(f"2 latent directions ≈ {lsa.explained_variance_ratio_.sum():.1%} of the matrix "
      "(the wb_02 map below is this, verified from first principles in the appendix)")

**Figures (regenerated in the appendix):**

<img src="../figures/wb_01_report_vector.png" width="700"/>
<img src="../figures/wb_02_document_map.png" width="700"/>
<img src="../figures/wb_03_word_map.png" width="700"/>

> [!note] The punchline of Act 1
> The map above is a **latent space**: meaning encoded as position, learned
> from word co-occurrence alone. Keep it in view all afternoon — in Act 6 a
> GPT will *learn* vectors like these from our narratives by gradient descent
> (`nn.Embedding` — the very line from last week's `gpt_train.py`).
>
> *(Depth marker: our map is LSA — truncated SVD of the idf-weighted presence
> matrix; Eckart–Young makes it the optimal rank-k compression. `briefing-0b`.)*

---
# Act 2 · Why we regularize our features — and how it works
Act 1 left every report as a point with 6,907 coordinates. A model pays
**estimation variance** for every coefficient it fits — and coefficients on
noise columns never repay it. Regularization is the discipline that manages
that budget: **keep the features, shrink the coefficients** — deliberately
trading a little bias for a lot of variance. Two pictures carry the whole
theory: the constraint geometry (why L1 zeroes coefficients *exactly* —
feature selection arriving as a by-product; the seed of Act 3) and the
measured bias–variance trade. Derivations: `briefing-2` (soft-thresholding
from subgradients; ridge as SVD shrinkage).

*Watch for it in Act 6:* the class's GPT quietly used both modern regularizers
— `dropout = 0.1` and the **W** in `AdamW` (weight decay).

In [ ]:
# ⚙ Act 2 lead — the L1 path with the standard API: C = 1/λ controls sparsity
# (C=0.05 -> 12 words, C=1.0 -> 278: verified 2026-07-01; ~2-3 min for the CV column)
yfire_s = yfire[m12].astype(bool)                 # fire-vs-rest, as in wb_16
for C_ in [0.01, 0.05, 0.2, 1.0]:
    l1 = Pipeline([("tfidf", TfidfVectorizer(min_df=2)),
                   ("clf", LogisticRegression(penalty="l1", solver="liblinear",
                                              C=C_, random_state=0))])
    f1 = cross_val_score(l1, X_text, yfire_s, cv=cv, scoring="f1", n_jobs=-1).mean()
    l1.fit(X_text, yfire_s)
    nz = np.flatnonzero(l1.named_steps["clf"].coef_.ravel())
    print(f"C={C_:<5} kept {len(nz):>4} of {l1.named_steps['clf'].coef_.shape[1]:,} words   F1={f1:.3f}")

In [ ]:
# ⚙ sklearn regularizes BY DEFAULT — and so did your GPT
prm = LogisticRegression().get_params()
print({k: prm[k] for k in ("penalty", "C")}, "<- on unless you turn it off (C = 1/λ)")
print("And so did your GPT: dropout=0.1 + AdamW weight decay (the W).")
print("(AdamW's decay value: confirm from gpt_run_outputs/*_meta.txt.)")

**Figures (regenerated in the appendix):**

<img src="../figures/wb_14_l1_l2_geometry.png" width="700"/>
<img src="../figures/wb_15_bias_variance.png" width="700"/>
<img src="../figures/wb_16_l1_path.png" width="700"/>

---
# Act 3 · Which features matter?
Act 2 shrank coefficients and got zeros as a by-product. This act attacks the
title question head-on: of the TF-IDF matrix's 6,907 columns (24,668
documents), *which carry signal about the target?* Most are noise (`yof`,
`the`) or near-duplicates — keeping them costs estimation variance (Act 2's
ledger) and buys nothing. Three families of evidence (Briefing 1):

| family | idea | cost |
| --- | --- | --- |
| **filter** | score each feature vs target (χ², MI, F) — model-free | ~free |
| **embedded** | the model selects *while* it fits (L1 — Act 2, just seen) | one fit |
| wrapper | retrain on subsets (RFE) | expensive — skipped |

χ² in one sentence: *how far is the observed word-class table from what
independence predicts, in units of sampling noise?* — and the same scores will
apply, unchanged, to learned embedding dimensions in Act 6.

In [ ]:
# ⚙ Act 3 lead — which words matter? chi2-rank all features: two lines of API
sel = SelectKBest(chi2, k=2000).fit(Xt, y_lab)
for j in np.argsort(sel.scores_)[::-1][:10]:
    print(f"{tfidf.get_feature_names_out()[j]:<12} chi2 = {sel.scores_[j]:>9,.0f}")

### ⚙ The production version (scikit-learn, run locally)
Same experiment with the real pipeline — TF-IDF, χ² selector **inside** the CV
folds (leak-proof by construction), logistic regression. Expected: baseline ≈
**0.856**; k=1000 → 0.8552; k=2000 → **0.8567** (verified 2026-07-01,
sklearn 1.9). No English stop list — it deletes `fire` (27% of narratives).

In [ ]:
# ⚙ the selection sweep (the demo's spine): does deleting 85% of columns cost anything?
for k in [1000, 2000, "all"]:
    pipe = Pipeline([("tfidf", TfidfVectorizer(min_df=2)),
                     ("select", SelectKBest(chi2, k=k)),
                     ("clf", LogisticRegression(max_iter=2000))])
    s = cross_val_score(pipe, X_text, y_lab, cv=cv, scoring="f1_macro", n_jobs=-1)
    print(f"k={str(k):>5}: macro-F1 = {s.mean():.4f} +/- {s.std(ddof=1)/np.sqrt(5):.4f}")

**Figures (regenerated in the appendix):**

<img src="../figures/wb_12_chi2.png" width="700"/>
<img src="../figures/wb_13_selection_knee.png" width="700"/>

---
# Act 4 · How do we tune a model — honestly?
Training loss always votes for *more capacity, less penalty* — it would set
every knob to "memorize." The only honest referee is **data the fit never
saw**: cross-validation. Metric note for 12 imbalanced classes: always-majority
scores accuracy 0.229 but **macro-F1 0.031** (real numbers, this data) — so
every score here is macro-F1. Then: tune the knobs (grid vs random, the one-SE
rule), bake off model *types* on identical folds, and close with the two
leakage crimes that make great-looking models worthless.

In [ ]:
# ⚙ Act 4 lead — tuning with the standard tool: GridSearchCV (~2-3 min)
# (RandomizedSearchCV = same API, param distributions instead of a grid — wb_17's point)
from sklearn.model_selection import GridSearchCV
pipe = Pipeline([("tfidf", TfidfVectorizer(min_df=2)),
                 ("select", SelectKBest(chi2, k=2000)),
                 ("clf", LogisticRegression(max_iter=2000))])
gs = GridSearchCV(pipe, {"clf__C": [0.1, 1.0, 10.0]}, cv=cv,
                  scoring="f1_macro", n_jobs=-1).fit(X_text, y_lab)
print(f"best C = {gs.best_params_['clf__C']}   macro-F1 = {gs.best_score_:.4f}")
print("(C=1.0 row should reproduce the sweep's k=2000 number ≈ 0.8567 — same pipeline)")

**Figures (regenerated in the appendix):**

<img src="../figures/wb_17_grid_vs_random.png" width="700"/>
<img src="../figures/wb_18_tuning_curve.png" width="700"/>

### ⚙ The bake-off: several model *types*, identical folds
Model selection in its **supporting** role: CV + macro-F1 is the *evidence*
that a feature set / model choice helped. Same folds for every model, mean ±
SE, one-SE rule. *(Honesty caveat: the winner's CV score is slightly
optimistic — the max of noisy estimates. Plain CV to choose; nested CV to
publish. Briefing 3+4 §4.3.)*

In [ ]:
# ⚙ sklearn bake-off + figure from its own results
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
models = {"logistic": LogisticRegression(max_iter=2000),
          "linear-SVM": LinearSVC(dual="auto", random_state=0),
          "complement-NB": ComplementNB(),
          "decision-tree": DecisionTreeClassifier(max_depth=30, random_state=0)}
names, ms_, es_ = [], [], []
for name, clf in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(min_df=2)),
                     ("select", SelectKBest(chi2, k=2000)), ("clf", clf)])
    s = cross_val_score(pipe, X_text, y_lab, cv=cv, scoring="f1_macro", n_jobs=-1)
    names.append(name); ms_.append(s.mean()); es_.append(s.std(ddof=1)/np.sqrt(5))
    print(f"{name:<15} macro-F1 = {s.mean():.4f} +/- {es_[-1]:.4f}")
order_ = np.argsort(ms_)
fig, ax = plt.subplots(figsize=(9.5, 5))
ax.barh([names[i] for i in order_], [ms_[i] for i in order_],
        xerr=[es_[i] for i in order_], color=OK[0], capsize=5)
thr = max(ms_) - es_[int(np.argmax(ms_))]
ax.axvline(thr, color=OK[1], ls="--", lw=2)
ax.text(thr, -0.45, " one SE below best", color=OK[1], fontsize=12)
ax.set_xlabel("5-fold CV macro-F1")
ax.set_title("the bake-off: identical folds, mean ± SE — pick on evidence")
SAVE(fig, "19_bakeoff"); plt.show()

### The gotcha that ends careers: leakage
Two crimes, one lesson — *any* data-dependent step outside the folds poisons
the referee. **(a)** `victim_1_severity` scores ~95% because the narrative
*states the outcome* (92% of Death narratives contain a death-word;
P(Death | death-word) = 98% — verified 2026-06-25). **(b)** Selecting features
before CV: on **coin-flip labels** it reports 0.815; the honest pipeline says
0.515 (verified 2026-07-02).

In [ ]:
# ⚙ leakage, live: select-then-CV vs Pipeline-in-CV on PURE NOISE labels
rng4 = np.random.default_rng(0)
ynoise = rng4.integers(0, 2, size=3000)
Xn = TfidfVectorizer(min_df=2).fit_transform(X_text.iloc[:3000])
leaky_sel = SelectKBest(chi2, k=100).fit(Xn, ynoise)          # sees ALL labels
leaky = cross_val_score(LogisticRegression(max_iter=2000),
                        leaky_sel.transform(Xn), ynoise, cv=5)
honest = cross_val_score(Pipeline([("select", SelectKBest(chi2, k=100)),
                                   ("clf", LogisticRegression(max_iter=2000))]),
                         Xn, ynoise, cv=5)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(["select THEN cross-validate\n(the crime)",
               "selection INSIDE the folds\n(the truth)"],
              [leaky.mean(), honest.mean()], color=[OK[1], OK[0]], width=0.55)
ax.axhline(0.5, color="gray", ls=":", lw=2)
ax.text(1.32, 0.51, "chance", color="gray", fontsize=12)
for b_, v_ in zip(bars, [leaky.mean(), honest.mean()]):
    ax.text(b_.get_x()+b_.get_width()/2, v_+0.015, f"{v_:.3f}", ha="center",
            fontsize=15, weight="bold")
ax.set_ylim(0, 1); ax.set_ylabel("CV accuracy on COIN-FLIP labels")
ax.set_title("labels are pure noise — one workflow finds 'signal' anyway")
SAVE(fig, "20_leakage"); plt.show()

---
# Act 5 · What is a neural network, really?
Strip the mystique in three figures: a **neuron** is a weighted sum squashed to
a probability — *which is exactly the logistic regression you have fit for
years* (the weights below are real, from Act 2's L1 path). A **network** is
neurons wired in layers so the boundary can bend. **Training** is walking
downhill on a loss surface — `loss.backward()` computes the slope (chain rule
= backpropagation), `optimizer.step()` takes the step. Nothing you don't
already own; the frontier is these three ideas, composed.

*This act is deliberately library-free: the point is that there is nothing in
the box a stats room doesn't already own. The hand-coded neuron, backprop, and
descent live in the appendix — these figures are their output.*

<img src="../figures/wb_04_neuron.png" width="700"/>
<img src="../figures/wb_05_why_layers.png" width="700"/>
<img src="../figures/wb_06_loss_surface.png" width="700"/>

> [!note] Act 5 wrap — three sentences to say on stage
> 1. A neuron is logistic regression; you have fit these for years.
> 2. Layers let the boundary bend; depth lets simple parts compose into
>    features — in text: characters → syllables → words → meaning.
> 3. Training never changes: define a loss, follow its gradient downhill. In
>    `gpt_train.py` that's `loss.backward()`, `optimizer.step()`,
>    `optimizer.zero_grad()` — repeated 3,000 times. Next act: the machine
>    itself.

---
# Act 6 · A CPSC GPT — the class's model, retrained on our own narratives
Last week you ran `gpt_train.py` on `article.txt` — code only, no concepts.
Today's artifact: **the identical architecture (same hyperparameters, same
seed), retrained once, offline, on 29.1 million characters of CPSC incident
narratives** — `python gpt_train_instrumented.py --from-cpsc` (one offline run — **done 2026-07-13**).
It learns to write *fake incident reports*. By the end of this act you will be
able to explain every line of the script.

### One architecture, two data regimes

| run | corpus | vocab | parameters | params / training char |
| --- | ---: | ---: | ---: | ---: |
| class (`article.txt`) | 29,961 chars | 82 | 829,266 | **~31** |
| **CPSC GPT** (2026-07-13 run) | 29,077,970 chars | 109 | 836,205\* | **~0.032** |

Same model, a **~1,000× difference in data pressure**. At ~31 params/char the
model *will* memorize (Act 2's variance argument, live); at 0.032 it is forced
to generalize. Dropout and AdamW's weight decay are the script's built-in
Act 2 — this table is why they mattered more for the class's run than for ours.

\* *computed from the architecture (13,952 + 16,384 + 4×197,888 + 256 + 14,061;
each vocab char costs 257 params) — **confirm against `cpsc_meta.txt`**.*

<img src="../figures/wb_07_gpt_architecture.png" width="560"/>

*(drawn to exact param counts by the appendix cell)*

### The embedding table — Act 1's map, learned per character
`nn.Embedding(109, 128)` is literally a **table: 109 rows (characters) × 128
columns**. "Embedding a token" = look up its row. Training nudges the rows so
characters used in similar contexts get similar rows — the same geometry as the
CPSC word map, discovered by gradient descent instead of counting.

In [ ]:
# FIG 8 — the embedding table (init vs trained-if-available)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
Wemb0 = rng.normal(0, 0.02, (109, 128))                 # exactly the script's init
axes[0].imshow(Wemb0, aspect="auto", cmap="RdBu_r", vmin=-0.06, vmax=0.06)
axes[0].set_title("at initialization: 109×128 of noise\n(std=0.02, like _init_weights)", fontsize=13)
axes[0].set_xlabel("128 embedding dimensions"); axes[0].set_ylabel("109 characters")
try:                                                   # after the one offline re-run
    import torch
    ck = torch.load("gpt_run_outputs/cpsc_ckpt.pt", map_location="cpu")
    Wt = ck["model_state"]["token_embedding.weight"].numpy()
    itos = ck["itos"]
    axes[1].imshow(Wt, aspect="auto", cmap="RdBu_r",
                   vmin=-np.abs(Wt).max()*.6, vmax=np.abs(Wt).max()*.6)
    axes[1].set_title("after 3,000 steps: structure appears (REAL, your run)", fontsize=13)
    # nearest neighbours of a vowel, in the learned space:
    q = Wt / np.linalg.norm(Wt, axis=1, keepdims=True)
    i_e = [i for i, ch in itos.items() if ch == "E"][0]
    sims = q @ q[i_e]
    nn5 = [repr(itos[i]) for i in np.argsort(-sims)[1:6]]
    print("nearest characters to 'E' in the LEARNED space:", nn5)
except Exception:
    axes[1].text(0.5, 0.5, "run gpt_train_instrumented.py --from-cpsc\n→ this panel shows the CPSC GPT\u2019s"
                 "\ntrained table + neighbor chars", ha="center", va="center",
                 fontsize=14, transform=axes[1].transAxes)
    axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_xlabel("128 embedding dimensions")
SAVE(fig, "08_embedding_table"); plt.show()

### Attention — the only genuinely new idea in the stack
Each position builds its representation as a **weighted average of earlier
positions**, with weights the model *learns to compute* from the content
(query·key). Intuition: to guess the character after `"THE STO_"`, the model
learns to look back at `S-T-O` (and further, at `FIRE`-ish context).
Two constraints in your script: the **causal mask** (`torch.tril`) — no peeking
at the future — and $\sqrt{d_k}$ scaling (variance control; verified
numerically in `briefing-0b`).

<img src="../figures/wb_09_attention.png" width="700"/>

In [ ]:
# FIG 10 — what the 45 minutes bought: the training curve
# Before training: pure guessing over 109 chars -> cross-entropy = ln(109)
print(f"loss before any learning = ln(109) = {np.log(109):.3f} nats/char")
fig, ax = plt.subplots(figsize=(10, 6))
try:
    hist = pd.read_csv("gpt_run_outputs/cpsc_losses.csv")        # the real re-run
    ax.plot(hist["step"], hist["train_loss"], lw=3, marker="o", color=OK[0], label="train")
    ax.plot(hist["step"], hist["val_loss"], lw=3, marker="s", color=OK[1], label="val")
    ax.set_title("the CPSC GPT: what 3,000 steps on a CPU bought", fontsize=16)
except FileNotFoundError:
    stp = np.linspace(0, 3000, 60)                                # SCHEMATIC until re-run
    ax.plot(stp, 1.15 + 3.26*np.exp(-stp/450), lw=3, color=OK[0], label="train (schematic)")
    ax.plot(stp, 1.55 + 2.86*np.exp(-stp/500) + stp/3000*0.25, lw=3, color=OK[1],
            label="val (schematic)")
    ax.set_title("SCHEMATIC until the instrumented re-run — shapes, not data",
                 fontsize=16, color=OK[1])
ax.axhline(np.log(109), color="gray", ls=":", lw=2)
ax.text(1500, np.log(109)+0.06, "ln(109) = 4.69 — random guessing", fontsize=12,
        color="gray", ha="center")
ax.set_xlabel("training step"); ax.set_ylabel("cross-entropy (nats/char)")
ax.legend(fontsize=14); ax.grid(alpha=0.3)
SAVE(fig, "10_loss_curve"); plt.show()
print("At 0.032 params per training char expect a small train-val gap; the")
print("class's ~31/char article run is the memorization regime — Act 2, live.")

<img src="../figures/wb_11_temperature.png" width="700"/>

*(live sampling from the checkpoint: `gpt_sampling.ipynb` — rehearsed, with fallback)*

In [ ]:
# the payoff: what the CPSC GPT writes (fake incident reports!)
try:
    txt = open("gpt_run_outputs/cpsc_samples.txt", encoding="utf-8").read()
    for block in txt.split("--- ")[1:]:
        head, _, body = block.partition("\n")
        print(f"=== {head.strip(' -')} ===")
        print(body.strip()[:420], "…\n")
except FileNotFoundError:
    print("Run `python gpt_train_instrumented.py --from-cpsc` once (~45 min) —")
    print("then this cell prints the model's INVENTED incident narratives at")
    print("three temperature settings. (They are fluent, in-style, and factually")
    print("empty — the 'base model' beat, in the audience's own domain.)")

> [!note] The "base model" beat — and the close (~60 s)
> The CPSC GPT produces *fluent fake incident reports with no facts behind
> them* — exactly what it was asked to learn: **next-character prediction,
> nothing else**. A model trained this way is a **base model**. GPT-4-class
> systems are this same recipe at ~10⁶× scale *plus* instruction tuning
> afterwards. Nobody in this room trained a chatbot last week; you trained the
> thing chatbots are made from — and now you've watched one learn our domain's
> style from scratch.
>
> *Close, on stage:* "Last week you trained a GPT and it felt like magic.
> There is no magic: its embeddings are Act 1's map, learned instead of
> counted; its dropout and weight decay are Act 2; its training is Act 5's
> descent; its train/val split is Act 4's referee. One discipline ran through
> everything this afternoon — decide which features matter, and prove it with
> held-out evidence — and it is the same discipline whether your features are
> TF-IDF columns or the 128 dimensions your GPT just learned.\"

---
## Take-home exercises
1. **Find the knee** — extend Act 3's k-grid to [100, 600, 4000]; where does
   the curve flatten, and what does the one-SE rule pick?
2. **Principled disagreement** — swap χ² → mutual information (binary presence
   + `discrete_features=True`; see Briefing 1 for the sparse-input trap). Diff
   the top-50 lists; find `monoxide`.
3. **Regularization as selection** — extend the ⚙ L1 cell to
   C ∈ [0.01, 0.03, 0.1, 0.3, 1, 3]; plot nonzero-count and F1 vs C.
4. **Feel the leak** — in the leakage cell, raise k to 500 and watch the fake
   signal grow.
5. **Read your own GPT** — in `gpt_train.py`, highlight the regularization
   lines, the hyperparameter block, and the evaluation split. Hand-count the
   parameters from the config (CPSC run: 836,205 — confirm against your
   `cpsc_meta.txt`; the class's article run: 829,266 — the difference is only
   the vocab size, 257 params per character).
6. **Inference-time knobs** — `python gpt_train_instrumented.py
   --generate-only gpt_run_outputs/cpsc_ckpt.pt --temperature 0.6 --top-k 20`,
   then 1.3 / 10. Reconcile what you see with the temperature figure.

*Theory & derivations: `curriculum/briefing-0*…3-4*.md`. Every number here traces to code in this
repo.*

---
# Appendix · Under the hood — figure factory & first-principles receipts
Everything below is **numpy-only, on purpose**: each cell *derives* a statistic
the acts used via the standard API, and regenerates the corresponding
`wb_*.png`. Nothing here is code you'd write at work — that's the point: the
API calls above are not incantations, and every figure on the slides has a
receipt. Run before slide freeze; skip on stage.

In [ ]:
# FIG 1 — one real incident report -> its vector of word evidence
i0 = int(np.argmax(X[:, w2i["fire"]] * X[:, w2i["stove"]]))     # a fire narrative
txt = " ".join(str(df["incident_description"].iloc[i0]).split())
print("REPORT:", txt[:300], "…")
present = [w for w in vocab if X[i0, w2i[w]]]
wts = sorted(((w, idf[w2i[w]]) for w in present), key=lambda t: -t[1])[:14]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh([w for w, _ in wts][::-1], [v for _, v in wts][::-1], color=OK[0])
ax.set_xlabel("weight (idf: rarer word = more informative)")
ax.set_title("One report → one row of numbers\n(its most informative words)")
SAVE(fig, "01_report_vector"); plt.show()

In [ ]:
# FIG 2 — every report is a point; similar words -> similar place (real data)
pick = [c_ for c_ in ["Thermal - Fire", "Mechanical - Fall", "Mechanical - Submersion",
        "Electrical - Other Electrical malfunction"] if (df["product_1_hazard"] == c_).sum() > 150]
mask = df["product_1_hazard"].isin(pick).values
Xs = Xw[mask]; lab = df.loc[mask, "product_1_hazard"].values
sub = rng.choice(len(Xs), min(4000, len(Xs)), replace=False)
Xs, lab = Xs[sub], lab[sub]
Xc = Xs - Xs.mean(0)
ev, V = np.linalg.eigh(Xc.T @ Xc)               # PCA via eigen-decomposition
P = Xc @ V[:, -2:]
fig, ax = plt.subplots(figsize=(10, 7))
for cl, col in zip(pick, OK):
    m = lab == cl
    ax.scatter(P[m, 0], P[m, 1], s=8, alpha=0.45, color=col,
               label=cl.split(" - ")[-1].split(" ")[-1 if "Electrical" in cl else 0])
ax.legend(fontsize=13, markerscale=2.5, framealpha=0.9)
ax.set_xlabel("latent direction 1"); ax.set_ylabel("latent direction 2")
ax.set_title("4,000 real incident reports, projected to 2 latent directions\n"
             "— no labels were used to draw this; the geometry alone separates them")
ax.set_xticks([]); ax.set_yticks([])
SAVE(fig, "02_document_map"); plt.show()

In [ ]:
# FIG 3 — WORDS as vectors: geometry = meaning, learned from CPSC alone
words = ["fire","flame","smoke","stove","candle","grill","lighter","burned",
         "pool","water","drowning","drowned","swimming","bathtub",
         "fell","fall","floor","stairs","ladder","fracture","hip",
         "carbon","monoxide","fumes","poisoning",
         "shock","electrical","electric","outlet","cord","wire","battery"]
words = [w for w in words if w in w2i]
E = (V[:, -30:].T)[:, [w2i[w] for w in words]].T     # words in 30 latent dims
E = E / np.linalg.norm(E, axis=1, keepdims=True)
Ec = E - E.mean(0)
_, Vw = np.linalg.eigh(Ec.T @ Ec)
Q = Ec @ Vw[:, -2:]
fig, ax = plt.subplots(figsize=(10, 7.5))
ax.scatter(Q[:, 0], Q[:, 1], s=14, color="#666666")
for i, w in enumerate(words):
    ax.annotate(w, (Q[i, 0], Q[i, 1]), fontsize=13, xytext=(4, 4),
                textcoords="offset points")
ax.set_title("Words as vectors — nearby = related\n"
             "(pool/swimming/drowned; fell/fracture/hip; carbon/monoxide — "
             "learned from co-occurrence alone)")
ax.set_xticks([]); ax.set_yticks([])
SAVE(fig, "03_word_map"); plt.show()

In [ ]:
# FIG 14 — THE picture: same loss, two constraint shapes (why L1 selects)
tg = np.linspace(0, 2*np.pi, 400)
b_star = np.array([1.7, 1.1]); Am = np.array([[1.0, 0.45], [0.45, 0.55]])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 6))
for axx, is_l1 in zip(axes, [False, True]):
    vals, vecs = np.linalg.eigh(Am)
    for r in [0.25, 0.7, 1.3, 2.0, 2.85]:
        ell = (vecs @ np.diag(np.sqrt(r/vals)) @ np.c_[np.cos(tg), np.sin(tg)].T).T + b_star
        axx.plot(ell[:, 0], ell[:, 1], color=OK[0], lw=1.4, alpha=0.75)
    if is_l1:
        t_r = 0.9
        axx.fill([t_r, 0, -t_r, 0], [0, t_r, 0, -t_r], color=OK[1], alpha=0.35)
        axx.plot([t_r, 0, -t_r, 0, t_r], [0, t_r, 0, -t_r, 0], color=OK[1], lw=2.5)
        axx.plot(0, t_r, "o", ms=13, color="black")
        axx.annotate("solution AT the corner:\nβ₁ = 0 exactly", (0, t_r),
                     xytext=(-2.7, 2.1), fontsize=13, weight="bold",
                     arrowprops=dict(arrowstyle="->", lw=1.5))
        axx.set_title("L1 (lasso): corners → exact zeros", fontsize=15)
    else:
        t_r = 0.95
        axx.plot(t_r*np.cos(tg), t_r*np.sin(tg), color=OK[2], lw=2.5)
        axx.fill(t_r*np.cos(tg), t_r*np.sin(tg), color=OK[2], alpha=0.3)
        axx.plot(0.52, 0.80, "o", ms=13, color="black")
        axx.annotate("solution: shrunken,\nboth coords ≠ 0", (0.52, 0.80),
                     xytext=(-2.7, 2.1), fontsize=13, weight="bold",
                     arrowprops=dict(arrowstyle="->", lw=1.5))
        axx.set_title("L2 (ridge): smooth → no zeros", fontsize=15)
    axx.plot(*b_star, "*", ms=18, color=OK[0])
    axx.annotate("unpenalized fit", b_star, xytext=(b_star[0]+0.15, b_star[1]+0.3), fontsize=12)
    axx.axhline(0, color="gray", lw=.7); axx.axvline(0, color="gray", lw=.7)
    axx.set_xlim(-3, 3.4); axx.set_ylim(-1.7, 3.1); axx.set_aspect("equal")
    axx.set_xlabel("β₁"); axx.set_ylabel("β₂")
fig.suptitle("SCHEMATIC — grow the loss contours until they touch the budget shape",
             fontsize=15)
SAVE(fig, "14_l1_l2_geometry"); plt.show()
print("Derivation (soft-thresholding from subgradients): briefing-2 §4.")

In [ ]:
# FIG 15 — the trade measured + the two shrinkage operators (REAL simulation)
n_, p_, s2 = 60, 30, 4.0
beta = np.zeros(p_); beta[:5] = [2, -2, 1.5, -1.5, 1]
Xsim = np.random.default_rng(0).normal(size=(n_, p_)); Xsim -= Xsim.mean(0)
lams = np.array([0.001, 0.5, 2, 8, 32, 128, 512]); R = 400
rng2 = np.random.default_rng(1)
B = np.zeros((len(lams), R, p_))
for r in range(R):
    ysim = Xsim @ beta + rng2.normal(0, np.sqrt(s2), n_)
    for i, lam in enumerate(lams):
        B[i, r] = np.linalg.solve(Xsim.T @ Xsim + lam*np.eye(p_), Xsim.T @ ysim)
bias2 = ((B.mean(1) - beta)**2).sum(1)
var = B.var(1).sum(1)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))
axL = axes[0]
axL.plot(lams, bias2, "-o", lw=3, ms=8, color=OK[1], label="bias²")
axL.plot(lams, var, "-s", lw=3, ms=8, color=OK[0], label="variance")
axL.plot(lams, bias2 + var, "-D", lw=3, ms=8, color="black", label="MSE = bias² + var")
axL.set_xscale("log"); axL.set_xlabel("λ (penalty strength)")
axL.annotate("sweet spot", (8, (bias2+var)[3]), xytext=(20, 3.6), fontsize=13,
             weight="bold", arrowprops=dict(arrowstyle="->", lw=1.5))
axL.legend(fontsize=13); axL.grid(alpha=0.3)
axL.set_title("the trade, measured (400-rep simulation)\nidentity bias²+var=MSE exact")
bols = np.linspace(-3, 3, 300)
axR = axes[1]
axR.plot(bols, bols, ls=":", color="gray", lw=2, label="no penalty")
axR.plot(bols, bols/(1+1.0), lw=3, color=OK[2], label="ridge: rescale by 1/(1+λ)")
axR.plot(bols, np.sign(bols)*np.maximum(np.abs(bols)-1.0, 0), lw=3, color=OK[1],
         label="lasso: subtract λ, clip at 0")
axR.axhspan(-0.02, 0.02, xmin=0.335, xmax=0.665, color=OK[1], alpha=0.25)
axR.set_xlabel("unpenalized coefficient"); axR.set_ylabel("penalized coefficient")
axR.legend(fontsize=12); axR.grid(alpha=0.3)
axR.set_title("what each penalty does to a coefficient\n(L1's flat zone = feature removal)")
SAVE(fig, "15_bias_variance"); plt.show()

In [ ]:
# FIG 16 — the L1 path on CPSC, fire-vs-rest (REAL; FISTA ~2-4 min locally)
# Verified 2026-07-10: lam .03/.01/.001/.0003 -> 1/7/96/225 words, F1 .747/.775/.862/.887
FAST = False          # True: fewer iterations (shape identical, numbers approximate)
iters = 120 if FAST else 250
Xf = X; yf_ = yfire.astype(np.float32); nn_ = len(yf_)
v = np.random.default_rng(0).normal(size=1000).astype(np.float32)
for _ in range(20): v = Xf.T @ (Xf @ v); v /= np.linalg.norm(v)
L = float(v @ (Xf.T @ (Xf @ v)))/(4*nn_)
lams_ = [0.03, 0.01, 0.001, 0.0003]; nnz_, f1_, tops_ = [], [], []
w = np.zeros(1000, dtype=np.float32); b = 0.0
for lam in lams_:
    wz, bz, t = w.copy(), b, 1.0
    for _ in range(iters):
        pr = sigmoid(Xf @ wz + bz)
        w_new = wz - (Xf.T @ (pr - yf_)/nn_)/L
        w_new = np.sign(w_new)*np.maximum(np.abs(w_new) - lam/L, 0)
        b_new = bz - float((pr - yf_).mean())/L
        t_new = (1 + np.sqrt(1 + 4*t*t))/2
        wz = w_new + (t-1)/t_new*(w_new - w); bz = b_new + (t-1)/t_new*(b_new - b)
        w, b, t = w_new, b_new, t_new
    pr = sigmoid(Xf @ w + b); pred = pr > 0.5
    tp = int((pred & (yf_ == 1)).sum()); fp = int((pred & (yf_ == 0)).sum())
    fn = int((~pred & (yf_ == 1)).sum())
    nnz_.append(int((np.abs(w) > 1e-6).sum())); f1_.append(2*tp/(2*tp+fp+fn))
    top3 = np.argsort(-np.abs(w))[:3]
    tops_.append(", ".join(vocab[j2] for j2 in top3 if abs(w[j2]) > 1e-6))
fig, ax = plt.subplots(figsize=(10.5, 6))
ax.plot(nnz_, f1_, "-o", lw=3, ms=11, color=OK[0])
for x_, y_, t_ in zip(nnz_, f1_, tops_):
    ax.annotate(f"{x_} words\n({t_}…)", (x_, y_), xytext=(12, -22),
                textcoords="offset points", fontsize=12)
ax.set_xscale("log"); ax.set_xlabel("features kept (log)"); ax.set_ylabel("F1 (fire)")
ax.set_title("the L1 path: ONE word buys F1 0.75; diminishing returns after ~100\n"
             "(each point = one λ; L1 both fits and selects)")
ax.grid(alpha=0.3)
SAVE(fig, "16_l1_path"); plt.show()
print("Negative weights (electrical, cord, pool) = the model DISCOVERING that")
print("those narratives belong to other hazard classes.")

In [ ]:
# FIG 4 — one neuron, with REAL learned weights (Briefing 2's CPSC L1 path)
fig, ax = plt.subplots(figsize=(11, 6))
inputs = [("fire", 3.43, OK[1]), ("candle", 2.57, OK[1]), ("stove", 1.82, OK[1]),
          ("cord", -1.74, OK[0]), ("pool", -2.20, OK[0])]
for i, (w, wt, col) in enumerate(inputs):
    ypos = 5 - i * 1.1
    ax.add_patch(Circle((1, ypos), 0.32, fc="white", ec="black", lw=1.5))
    ax.text(1, ypos, w, ha="center", va="center", fontsize=13)
    ax.annotate("", xy=(4.7, 2.8), xytext=(1.35, ypos),
                arrowprops=dict(arrowstyle="-|>", color=col, lw=abs(wt),
                                alpha=0.85, shrinkA=0, shrinkB=14))
    ax.text(2.6, ypos + (2.8 - ypos) * 0.45 + 0.14, f"{wt:+.2f}",
            fontsize=12, color=col, weight="bold")
ax.add_patch(Circle((5, 2.8), 0.55, fc="#FFF2CC", ec="black", lw=2))
ax.text(5, 2.8, "Σ", ha="center", va="center", fontsize=24)
ax.annotate("", xy=(6.6, 2.8), xytext=(5.55, 2.8),
            arrowprops=dict(arrowstyle="-|>", color="black", lw=2))
ax.add_patch(Circle((7.15, 2.8), 0.55, fc="#DAE8FC", ec="black", lw=2))
ax.text(7.15, 2.8, "σ", ha="center", va="center", fontsize=22)
ax.annotate("", xy=(8.9, 2.8), xytext=(7.7, 2.8),
            arrowprops=dict(arrowstyle="-|>", color="black", lw=2))
ax.text(9.0, 2.8, "P(fire\nhazard)", fontsize=14, va="center")
ax.text(5, 1.35, "weighted sum of evidence", ha="center", fontsize=12, style="italic")
ax.text(7.15, 1.35, "squash to a\nprobability", ha="center", fontsize=12, style="italic")
ax.text(5.4, 5.75, "One neuron = logistic regression — real learned CPSC weights",
        ha="center", fontsize=15, weight="bold")
ax.set_xlim(0, 10.6); ax.set_ylim(0.05, 6.3); ax.axis("off")
SAVE(fig, "04_neuron"); plt.show()
print("thickness = |weight|; color = sign. 'pool' argues AGAINST fire — learned, not programmed.")

In [ ]:
# FIG 5 — why layers: one neuron draws a line; a hidden layer bends it
n = 240
th = rng.uniform(0, np.pi, n)
x1 = np.c_[np.cos(th), np.sin(th)] + rng.normal(0, 0.12, (n, 2))
x2 = np.c_[1 - np.cos(th), 0.4 - np.sin(th)] + rng.normal(0, 0.12, (n, 2))
Xm = np.vstack([x1, x2]); ym = np.r_[np.zeros(n), np.ones(n)]
Xm = (Xm - Xm.mean(0)) / Xm.std(0)
w = np.zeros(2); b = 0.0                              # logistic (one neuron)
for _ in range(3000):
    p = sigmoid(Xm @ w + b)
    w -= 0.5 * Xm.T @ (p - ym) / len(ym); b -= 0.5 * (p - ym).mean()
H = 8                                                 # tiny net: 2-8-1, trained by backprop
W1 = rng.normal(0, 1, (2, H)); b1 = np.zeros(H); W2 = rng.normal(0, 1, H); b2 = 0.0
for _ in range(4000):
    A1 = np.tanh(Xm @ W1 + b1); p = sigmoid(A1 @ W2 + b2)
    d2 = (p - ym) / len(ym)                           # chain rule, by hand:
    d1 = np.outer(d2, W2) * (1 - A1**2)               #   = backpropagation
    W2 -= 0.6 * A1.T @ d2; b2 -= 0.6 * d2.sum()
    W1 -= 0.6 * Xm.T @ d1; b1 -= 0.6 * d1.sum(0)
gx, gy = np.meshgrid(np.linspace(-2.6, 2.6, 220), np.linspace(-2.6, 2.6, 220))
G = np.c_[gx.ravel(), gy.ravel()]
Plin = sigmoid(G @ w + b).reshape(gx.shape)
Pnet = sigmoid(np.tanh(G @ W1 + b1) @ W2 + b2).reshape(gx.shape)
acc_lin = ((sigmoid(Xm @ w + b) > .5) == ym).mean()
acc_net = ((sigmoid(np.tanh(Xm @ W1 + b1) @ W2 + b2) > .5) == ym).mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))
for axx, Pm, t in zip(axes, [Plin, Pnet],
        [f"one neuron: a straight line — {acc_lin:.0%}",
         f"+ one hidden layer (8 neurons) — {acc_net:.0%}"]):
    axx.contourf(gx, gy, Pm, levels=[0, .5, 1], colors=["#D6E8F7", "#FBE3D6"], alpha=.9)
    axx.contour(gx, gy, Pm, levels=[.5], colors="black", linewidths=2)
    axx.scatter(*Xm[ym == 0].T, s=14, color=OK[0]); axx.scatter(*Xm[ym == 1].T, s=14, color=OK[1])
    axx.set_title(t, fontsize=15); axx.set_xticks([]); axx.set_yticks([])
fig.suptitle("Why layers: stacked neurons bend the boundary", fontsize=17)
SAVE(fig, "05_why_layers"); plt.show()

In [ ]:
# FIG 6 — training IS gradient descent on a loss surface (real CPSC, 2 weights)
sub6 = rng.choice(N, 6000, replace=False)
Xg = np.c_[X[sub6, w2i["fire"]], X[sub6, w2i["smoke"]]].astype(float)
yg = yfire[sub6]
def loss_at(w1, w2):
    p = sigmoid(Xg @ np.array([w1, w2]) - 1.8)        # intercept fixed for a 2-D picture
    return -(yg*np.log(p+1e-9) + (1-yg)*np.log(1-p+1e-9)).mean()
w1g, w2g = np.meshgrid(np.linspace(-1, 5, 90), np.linspace(-1, 5, 90))
Lg = np.array([[loss_at(a, c_) for a in w1g[0]] for c_ in w2g[:, 0]])
wv = np.array([-0.5, 4.5]); path = [wv.copy()]
for _ in range(60):
    p = sigmoid(Xg @ wv - 1.8)
    wv = wv - 2.2 * (Xg.T @ (p - yg) / len(yg))       # ← this is optimizer.step()
    path.append(wv.copy())
path = np.array(path)
fig, ax = plt.subplots(figsize=(9, 7))
cs = ax.contourf(w1g, w2g, Lg, levels=24, cmap="Blues_r")
ax.contour(w1g, w2g, Lg, levels=12, colors="white", linewidths=.6, alpha=.6)
ax.plot(path[:, 0], path[:, 1], "-o", color=OK[1], lw=2.5, ms=4.5)
ax.annotate("start (random)", path[0], xytext=(path[0][0]-0.3, path[0][1]+0.35),
            fontsize=13, color=OK[1], weight="bold")
ax.annotate("minimum", path[-1], xytext=(path[-1][0]+0.35, path[-1][1]-0.25),
            fontsize=13, color=OK[1], weight="bold")
ax.set_xlabel("weight on 'fire'"); ax.set_ylabel("weight on 'smoke'")
ax.set_title("Training = descending the loss surface\n"
             "(real fire-vs-rest loss; loss.backward() gives the slope,\n"
             "optimizer.step() takes the step — 829,266-dim version ran on your laptop)")
fig.colorbar(cs, label="cross-entropy loss")
SAVE(fig, "06_loss_surface"); plt.show()

In [ ]:
# FIG 12 — chi2, seen: the 'fire' contingency + all 1,000 words scored
sc = chi2_scores(X.astype(bool), yfire)
j = w2i["fire"]
has = X[:, j].astype(bool)
cells = np.array([[(has & yfire).sum(), (has & ~yfire).sum()],
                  [(~has & yfire).sum(), (~has & ~yfire).sum()]])
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axL = axes[0]
axL.bar(["'fire' present", "'fire' absent"], [cells[0,0], cells[1,0]],
        color=OK[1], label="fire hazard")
axL.bar(["'fire' present", "'fire' absent"], [cells[0,1], cells[1,1]],
        bottom=[cells[0,0], cells[1,0]], color=OK[0], label="other hazard")
for i, (a, b) in enumerate(cells):
    axL.text(i, a/2, f"{a:,}", ha="center", color="white", fontsize=13, weight="bold")
    axL.text(i, a + b/2, f"{b:,}", ha="center", color="white", fontsize=13, weight="bold")
axL.legend(); axL.set_ylabel("documents")
axL.set_title(f"the evidence χ² measures — for 'fire': χ² = {sc[j]:,.0f}")
axR = axes[1]
axR.semilogy(np.sort(sc)[::-1], lw=3, color=OK[0])
axR.axvline(100, color=OK[1], ls="--", lw=2)
axR.text(115, sc.max()/10, "a few hundred words\ncarry nearly all the signal", fontsize=13)
axR.set_xlabel("word rank by χ²"); axR.set_ylabel("χ² (log scale)")
axR.set_title("all 1,000 words scored: steep head, empty tail")
SAVE(fig, "12_chi2"); plt.show()
top10 = np.argsort(-sc)[:10]
print("top-10 by chi2 (fire-vs-rest):", [vocab[t] for t in top10])
print("Depth markers (Briefing 1): chi2 rewards rare-decisive, MI rewards")
print("common-reliable — monoxide is chi2 #1 but MI #142 on the 12-class task;")
print("and G2 = 2N*MI exactly, so the two disagree only where Pearson's Taylor")
print("approximation breaks.")

In [ ]:
# FIG 13 — does selection help? macro-F1 vs number of words kept (REAL 5-fold CV)
sc12 = np.zeros(1000)
Xb12 = X[m12].astype(bool)
for k in range(12):
    sc12 += chi2_scores(Xb12, y12 == k)              # 12-class chi2 (sum of one-vs-rest)
order = np.argsort(-sc12)
idx = np.random.default_rng(0).permutation(len(y12)); folds = np.array_split(idx, 5)
Y1 = np.eye(12, dtype=np.float32)[y12]
X12f = np.hstack([X[m12], np.ones((m12.sum(), 1), dtype=np.float32)])
ks = [30, 100, 300, 1000]; means, ses = [], []
for k in ks:
    cols = np.r_[order[:k], 1000]
    s = []
    for i in range(5):
        te = folds[i]; tr = np.concatenate([folds[j2] for j2 in range(5) if j2 != i])
        A = X12f[tr][:, cols]
        W = np.linalg.solve(A.T @ A + 10*np.eye(k+1, dtype=np.float32), A.T @ Y1[tr])
        s.append(macro_f1(y12[te], (X12f[te][:, cols] @ W).argmax(1)))
    means.append(np.mean(s)); ses.append(np.std(s, ddof=1)/np.sqrt(5))
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.errorbar(ks, means, yerr=ses, lw=3, marker="o", ms=10, capsize=5, color=OK[0])
for k, m_ in zip(ks, means): ax.annotate(f"{m_:.3f}", (k, m_), xytext=(0, 10),
                                         textcoords="offset points", ha="center", fontsize=12)
ax.set_xscale("log"); ax.set_xlabel("k = words kept (of 1,000), ranked by χ²")
ax.set_ylabel("5-fold CV macro-F1")
ax.set_title("the knee: 300 well-chosen words ≈ all 1,000\n"
             "(sklearn full pipeline: k=1000→0.8552, k=2000→0.8567, all 6,907→0.8562)")
ax.grid(alpha=0.3)
SAVE(fig, "13_selection_knee"); plt.show()

In [ ]:
# FIG 17 — same budget of 9 evaluations: grid wastes, random covers
def score_fn(x): return np.exp(-((x - 0.73)**2)/0.02)     # only x matters
gx_ = np.array([1/6, 3/6, 5/6])
rng3 = np.random.default_rng(3)
rx, ry = rng3.uniform(size=9), rng3.uniform(size=9)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.8))
xs = np.linspace(0, 1, 300)
for axx, px, py, name in [(axes[0], np.repeat(gx_, 3), np.tile(gx_, 3), "grid 3×3"),
                          (axes[1], rx, ry, "random ×9")]:
    axx.fill_between(xs, 0, score_fn(xs)*0.22, color=OK[2], alpha=0.4)
    axx.scatter(px, py, s=130, color=OK[0], zorder=3, edgecolor="black")
    for x_ in px: axx.plot([x_, x_], [-0.06, -0.02], color=OK[1], lw=2.5)
    axx.text(0.73, 0.24, "the only knob\nthat matters", ha="center", fontsize=12,
             color=OK[2], weight="bold")
    best = score_fn(px).max()
    axx.set_title(f"{name}: best found = {best:.2f}", fontsize=15)
    axx.set_xlim(0, 1); axx.set_ylim(-0.08, 1.05)
    axx.set_xlabel("important hyperparameter"); axx.set_ylabel("inert hyperparameter")
fig.suptitle("grid tries 3 distinct values of what matters; random tries 9 "
             "(P(random wins) = 0.877, verified)", fontsize=14)
SAVE(fig, "17_grid_vs_random"); plt.show()

In [ ]:
# FIG 18 — a REAL tuning curve + the one-SE rule (5-fold CV on CPSC, ~1-2 min)
grams = []
for i in range(5):
    tr = np.concatenate([folds[j2] for j2 in range(5) if j2 != i])
    grams.append((X12f[tr].T @ X12f[tr], X12f[tr].T @ Y1[tr]))
def cv_ridge(lam):
    s = []
    for i in range(5):
        W = np.linalg.solve(grams[i][0] + lam*np.eye(1001, dtype=np.float32), grams[i][1])
        s.append(macro_f1(y12[folds[i]], (X12f[folds[i]] @ W).argmax(1)))
    s = np.array(s); return s.mean(), s.std(ddof=1)/np.sqrt(5)
lamg = [0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
res = [cv_ridge(l) for l in lamg]
mm = np.array([r[0] for r in res]); ss = np.array([r[1] for r in res])
best = mm.argmax()
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(lamg, mm, yerr=ss, lw=3, marker="o", ms=10, capsize=5, color=OK[0])
ax.axhspan(mm[best]-ss[best], mm[best]+ss[best], color=OK[2], alpha=0.2,
           label="within one SE of best")
onese = max(l for l, m_ in zip(lamg, mm) if m_ >= mm[best]-ss[best])
ax.axvline(onese, color=OK[1], ls="--", lw=2.5)
ax.text(onese*1.3, mm.min()+0.05, f"one-SE pick:\nλ = {onese:g}\n(most regularized\n"
        "indistinguishable\nfrom best)", fontsize=12, color=OK[1], weight="bold")
ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("5-fold CV macro-F1")
ax.set_title("a real tuning curve: broad forgiving plateau, then collapse\n"
             "(Act 2's U-curve, live on 22,087 CPSC reports)")
ax.legend(fontsize=12); ax.grid(alpha=0.3)
SAVE(fig, "18_tuning_curve"); plt.show()
for l, (m_, s_) in zip(lamg, res): print(f"lam={l:<8} {m_:.4f} +/- {s_:.4f}")

In [ ]:
# FIG 7 — the architecture, bottom to top, with real parameter counts
# (vocab 109: tok 109*128=13,952 · pos 16,384 · 4×197,888 · ln_f 256 ·
#  head 128*109+109=14,061 → Σ 836,205; confirm vs cpsc_meta.txt)
fig, ax = plt.subplots(figsize=(8.5, 11.5))
def box(x, y, wd, h, text, fc, fs=12):
    ax.add_patch(FancyBboxPatch((x, y), wd, h, boxstyle="round,pad=0.06",
                                fc=fc, ec="black", lw=1.4))
    ax.text(x + wd/2, y + h/2, text, ha="center", va="center", fontsize=fs)
def arrow(y0, y1, x=5.0, txt=None):
    ax.annotate("", xy=(x, y1), xytext=(x, y0),
                arrowprops=dict(arrowstyle="-|>", lw=2, color="black"))
    if txt: ax.text(x + 0.15, (y0+y1)/2, txt, fontsize=11, va="center")
box(2.4, 0.3, 5.2, 0.75, '"T" → sample the next character', "#E8F4E8")
arrow(1.55, 1.05, txt="softmax + temperature / top-k")
box(2.4, 1.55, 5.2, 0.8, "lm_head: Linear 128 → 109   14,061\n(= multinomial logistic regression!)", "#FBE3D6")
arrow(2.85, 2.35)
box(2.4, 2.85, 5.2, 0.65, "final LayerNorm   256", "#F0F0F0")
arrow(4.6, 3.5)
box(1.6, 4.7, 6.8, 2.55, "", "#EDF3FB")
box(2.4, 6.15, 5.2, 0.7, "feed-forward  128→512→128   131,712", "#DAE8FC")
box(2.4, 5.15, 5.2, 0.7, "causal self-attention (4 heads)   65,664", "#DAE8FC")
ax.text(5.0, 7.5, "TransformerBlock ×4   (197,888 params each)",
        fontsize=13, weight="bold", ha="center")
ax.text(8.28, 5.85, "residual +\nLayerNorm", fontsize=10, ha="center", style="italic")
arrow(8.33, 7.32)
box(2.4, 8.35, 5.2, 0.7, "+ positional embedding 128×128   16,384", "#FFF2CC")
arrow(9.58, 9.07)
box(2.4, 9.6, 5.2, 0.8, "token embedding 109×128   13,952\n(every char becomes a 128-dim vector)", "#FFF2CC")
arrow(10.93, 10.42)
box(2.4, 10.95, 5.2, 0.75, 'CPSC narratives — 29,077,970 chars, vocab 109\n"THE FIRE STARTED IN…"', "#E8F4E8")
ax.text(5.0, 12.05, "the CPSC GPT — 836,205 parameters, end to end",
        fontsize=16, weight="bold", ha="center")
ax.set_xlim(0.8, 9.6); ax.set_ylim(0, 12.5); ax.axis("off")
SAVE(fig, "07_gpt_architecture"); plt.show()

In [ ]:
# FIG 9 — the causal mask + what attention weights look like
s = "THE STOVE CAUGHT F"
chars_ = list(s); T = len(chars_)
q = rng.normal(size=(T, 8)); k = rng.normal(size=(T, 8))
A = (q @ k.T) / np.sqrt(8)
A[np.triu_indices(T, 1)] = -np.inf                    # the causal mask
Aw = np.exp(A - A.max(1, keepdims=True)); Aw = Aw / Aw.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 6))
axes[0].imshow(np.tril(np.ones((10, 10))), cmap="Greys", vmin=0, vmax=1.6)
axes[0].set_title("the causal mask (torch.tril):\nposition t may only see ≤ t", fontsize=14)
axes[0].set_xlabel("attends to position"); axes[0].set_ylabel("position")
im = axes[1].imshow(Aw, cmap="Blues", vmin=0)
axes[1].set_xticks(range(T)); axes[1].set_xticklabels(chars_, fontsize=11)
axes[1].set_yticks(range(T)); axes[1].set_yticklabels(chars_, fontsize=11)
axes[1].set_title("attention weights: each row sums to 1\n(ILLUSTRATIVE weights, one head)", fontsize=14)
fig.colorbar(im, ax=axes[1], shrink=0.8)
SAVE(fig, "09_attention"); plt.show()
print("Each row = 'where this position looks'. 4 heads = 4 such tables per layer,")
print("learned end-to-end. FFN then transforms each position; residuals+LayerNorm")
print("keep 4 stacked blocks trainable. (Depth for Q&A: attention is Nadaraya-")
print("Watson kernel smoothing — briefing-0b.)")

In [ ]:
# FIG 11 — temperature & top-k: the knobs on the OUTPUT distribution
z = np.array([3.2, 2.5, 1.9, 1.2, 0.8, 0.3, -0.2, -0.8])          # illustrative logits
labs = list("etaoins ")
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), sharey=True)
for axx, T_ in zip(axes, [0.6, 1.0, 1.3]):
    p = np.exp(z/T_); p = p/p.sum()
    axx.bar(labs, p, color=[OK[0]]*3 + ["#B0C4DE"]*5)
    axx.set_title(f"temperature = {T_}", fontsize=15)
    axx.set_ylim(0, 0.75)
axes[0].set_ylabel("P(next char)")
axes[0].text(4.6, 0.62, "low T:\nconfident,\nrepetitive", fontsize=12)
axes[2].text(4.6, 0.55, "high T:\ncreative,\nerror-prone", fontsize=12)
fig.suptitle("generate(): divide logits by T before softmax; top-k zeroes the tail "
             "(illustrative logits)", fontsize=14)
SAVE(fig, "11_temperature"); plt.show()